In [19]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, "..")

from src.utils import load_xpt

data_dir = Path("../data/raw/nhanes")

In [20]:
demo_i = load_xpt(data_dir / "2015-2016/DEMO_I.xpt")
demo_j = load_xpt(data_dir / "2017-2018/DEMO_J.xpt")

demo_i.shape, demo_j.shape

((9971, 47), (9254, 46))

In [21]:
cols = ["SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3"]

demo_i[cols].head()

,SEQN,RIDAGEYR,RIAGENDR,RIDRETH3
0,83732.0,62.0,1.0,3.0
1,83733.0,53.0,1.0,3.0
2,83734.0,78.0,1.0,3.0
3,83735.0,56.0,2.0,3.0
4,83736.0,42.0,2.0,4.0


In [22]:
demo_i["SEQN"].duplicated().sum(), demo_j["SEQN"].duplicated().sum()

(np.int64(0), np.int64(0))

In [23]:
set(demo_i.columns) - set(demo_j.columns), \
set(demo_j.columns) - set(demo_i.columns)

({'DMDHRAGE', 'DMDHRBR4', 'DMDHREDU', 'DMDHRMAR', 'DMDHSEDU'},
 {'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ'})

In [24]:
cols = [
    "SEQN", "RIDAGEYR", "RIAGENDR", "RIDRETH3",
    "WTINT2YR", "WTMEC2YR", "SDMVPSU", "SDMVSTRA"
]

demo_i = demo_i[cols].copy()
demo_j = demo_j[cols].copy()

In [25]:
diq_i = pd.read_sas(
    data_dir / "2015-2016/DIQ_I.xpt", format="xport"
)
diq_j = pd.read_sas(
    data_dir / "2017-2018/DIQ_J.xpt", format="xport"
)

diq_i.shape, diq_j.shape

((9575, 54), (8897, 54))

In [26]:
cols = ["SEQN", "DIQ010", "DIQ050", "DIQ070"]

diq_i[cols].head()

,SEQN,DIQ010,DIQ050,DIQ070
0,83732.0,1.0,1.0,1.0
1,83733.0,2.0,2.0,NaN
2,83734.0,1.0,2.0,2.0
3,83735.0,2.0,2.0,2.0
4,83736.0,2.0,2.0,NaN


In [27]:
diq_i["SEQN"].duplicated().sum(), diq_j["SEQN"].duplicated().sum()

(np.int64(0), np.int64(0))

In [28]:
diq_cols = ["SEQN", "DIQ010", "DIQ050", "DIQ070"]

nhanes_i = demo_i.merge(
    diq_i[diq_cols], on="SEQN", how="left", validate="one_to_one"
)

nhanes_j = demo_j.merge(
    diq_j[diq_cols], on="SEQN", how="left", validate="one_to_one"
)

nhanes_i.shape, nhanes_j.shape

((9971, 11), (9254, 11))

In [29]:
pd.DataFrame({
    "2015-2016": nhanes_i[diq_cols[1:]].isna().sum(),
    "2017-2018": nhanes_j[diq_cols[1:]].isna().sum()
})

,2015-2016,2017-2018
DIQ010,396,357
DIQ050,396,357
DIQ070,8456,7580


In [30]:
adults_i = nhanes_i.loc[nhanes_i["RIDAGEYR"] >= 18].copy()
adults_j = nhanes_j.loc[nhanes_j["RIDAGEYR"] >= 18].copy()

adults_i.shape, adults_j.shape

((5992, 11), (5856, 11))

In [32]:
cols = ["DIQ010", "DIQ050", "DIQ070"]

pd.DataFrame({
    "2015-2016": adults_i[cols].isna().sum(),
    "2017-2018": adults_j[cols].isna().sum()
})

,2015-2016,2017-2018
DIQ010,0,0
DIQ050,0,0
DIQ070,4524,4227


In [33]:
pd.crosstab(
    adults_i["DIQ010"],
    adults_i["DIQ070"].fillna("Missing")
)

DIQ070,1.0,2.0,7.0,9.0,Missing
DIQ010,,,,,
1.0,577,266,1,0,1
2.0,31,459,0,1,4520
3.0,32,100,0,0,0
9.0,0,1,0,0,3


In [34]:
pd.crosstab(
    adults_j["DIQ010"],
    adults_j["DIQ070"].fillna("Missing")
)

DIQ070,1.0,2.0,9.0,Missing
DIQ010,,,,
1.0,644,234,0,0
2.0,38,537,1,4224
3.0,33,140,1,0
9.0,0,1,0,3


In [37]:
adults_i = adults_i.merge(
    diq_i[["SEQN", "DIQ160"]],
    on="SEQN", how="left", validate="one_to_one"
)

adults_j = adults_j.merge(
    diq_j[["SEQN", "DIQ160"]],
    on="SEQN", how="left", validate="one_to_one"
)

In [38]:
for df in (adults_i, adults_j):
    eligible = df["DIQ010"].isin([1, 3]) | df["DIQ160"].eq(1)

    display(
        df.loc[
            eligible & df["DIQ070"].isna(),
            ["SEQN", "DIQ010", "DIQ160", "DIQ070"]
        ]
    )

,SEQN,DIQ010,DIQ160,DIQ070
4635,91416.0,1.0,NaN,NaN


,SEQN,DIQ010,DIQ160,DIQ070


In [39]:
for df in (adults_i, adults_j):
    eligible = df["DIQ010"].isin([1, 3]) | df["DIQ160"].eq(1)

    display(pd.Series({
        "skip_missing": (~eligible & df["DIQ070"].isna()).sum(),
        "eligible_missing": (eligible & df["DIQ070"].isna()).sum()
    }))

skip_missing        4523
eligible_missing       1
dtype: int64

skip_missing        4227
eligible_missing       0
dtype: int64

In [40]:
ghb_i = load_xpt(data_dir / "2015-2016/GHB_I.xpt")
ghb_j = load_xpt(data_dir / "2017-2018/GHB_J.xpt")

ghb_i.shape, ghb_j.shape

((6744, 2), (6401, 2))

In [41]:
ghb_i.columns.tolist(), ghb_j.columns.tolist()

(['SEQN', 'LBXGH'], ['SEQN', 'LBXGH'])

In [42]:
ghb_i["SEQN"].duplicated().sum(), ghb_j["SEQN"].duplicated().sum()

(np.int64(0), np.int64(0))

In [43]:
adults_i = adults_i.merge(
    ghb_i[["SEQN", "LBXGH"]],
    on="SEQN", how="left", validate="one_to_one"
)

adults_j = adults_j.merge(
    ghb_j[["SEQN", "LBXGH"]],
    on="SEQN", how="left", validate="one_to_one"
)

In [44]:
pd.DataFrame({
    "2015-2016": adults_i["LBXGH"].describe(),
    "2017-2018": adults_j["LBXGH"].describe()
})

,2015-2016,2017-2018
count,5441.000000,5261.000000
mean,5.822000,5.837179
std,1.165878,1.086560
min,3.800000,3.800000
25%,5.300000,5.300000
50%,5.500000,5.600000
75%,5.900000,6.000000
max,17.000000,16.200000


In [45]:
adults_i.shape, adults_j.shape, (
    adults_i["LBXGH"].isna().sum(),
    adults_j["LBXGH"].isna().sum()
)

((5992, 15), (5856, 15), (np.int64(551), np.int64(595)))

In [46]:
glu_i = load_xpt(data_dir / "2015-2016/GLU_I.xpt")
glu_j = load_xpt(data_dir / "2017-2018/GLU_J.xpt")

In [47]:
glu_i.columns.tolist(), glu_j.columns.tolist()

(['SEQN', 'WTSAF2YR', 'LBXGLU', 'LBDGLUSI'],
 ['SEQN', 'WTSAF2YR', 'LBXGLU', 'LBDGLUSI'])

In [48]:
pd.DataFrame({
    cycle: {
        "rows": len(df),
        "columns": len(df.columns),
        "SEQN_missing": df["SEQN"].isna().sum(),
        "SEQN_duplicates": df["SEQN"].duplicated().sum(),
        "LBXGLU_missing": df["LBXGLU"].isna().sum(),
    }
    for cycle, df in [("2015-2016", glu_i), ("2017-2018", glu_j)]
})

,2015-2016,2017-2018
rows,3191,3036
columns,4,4
SEQN_missing,0,0
SEQN_duplicates,0,0
LBXGLU_missing,219,145


In [49]:
glu_cols = ["SEQN", "LBXGLU", "WTSAF2YR"]

adults_i = adults_i.merge(
    glu_i[glu_cols],
    on="SEQN", how="left", validate="one_to_one"
)

adults_j = adults_j.merge(
    glu_j[glu_cols],
    on="SEQN", how="left", validate="one_to_one"
)

In [50]:
pd.DataFrame({
    cycle: {
        "adults": len(df),
        "LBXGLU_available": df["LBXGLU"].notna().sum(),
        "LBXGLU_missing": df["LBXGLU"].isna().sum(),
    }
    for cycle, df in [("2015-2016", adults_i), ("2017-2018", adults_j)]
})

,2015-2016,2017-2018
adults,5992,5856
LBXGLU_available,2573,2537
LBXGLU_missing,3419,3319


In [51]:
pd.DataFrame({
    cycle: {
        "glucose_available": df["LBXGLU"].notna().sum(),
        "glucose_with_positive_weight": (
            df["LBXGLU"].notna() & df["WTSAF2YR"].gt(0)
        ).sum(),
        "glucose_with_zero_weight": (
            df["LBXGLU"].notna() & df["WTSAF2YR"].eq(0)
        ).sum(),
        "glucose_with_missing_weight": (
            df["LBXGLU"].notna() & df["WTSAF2YR"].isna()
        ).sum(),
    }
    for cycle, df in [("2015-2016", adults_i), ("2017-2018", adults_j)]
})

,2015-2016,2017-2018
glucose_available,2573,2537
glucose_with_positive_weight,2573,2537
glucose_with_zero_weight,0,0
glucose_with_missing_weight,0,0
